In [ ]:
import numpy as np
# import ds_format as ds
import os
import glob
# import aquarius_time as aq
import xarray as xr
from scipy import interpolate
from netCDF4 import Dataset
from datetime import datetime
import pandas as pd
import time

import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
from matplotlib.ticker import MultipleLocator

import dask

data_dir = '/home/lukeew/Local_work/data/01_SLW_xgb'
vdr_mask_data_dir = data_dir + '/CHCH_vdr_mask/processed{}'

### Opening data for one day

In [ ]:
date = '20210605'
infile = vdr_mask_data_dir.format(date[:6]) + '/{}.nc'.format(date)
ds = xr.open_dataset(infile)

In [ ]:
ds

### Plotting VDR cloud mask

In [ ]:
fig, ax2 = plt.subplots(1,1)
Im2=ax2.imshow(ds['vdr_cm'][:].T,             
    extent=(0, 288, 0, 15),
    aspect='auto',
    origin='lower',
    vmin=0,
    vmax=1,
    interpolation='None',
    cmap='RdYlBu')
ax2.set_xticklabels([])
ax2.set_xticks(np.arange(36,288,36))
ax2.xaxis.set_minor_locator(MultipleLocator(12))
ax2.set_ylabel('Altitude (km)', fontsize=15)
cax2 = ax2.inset_axes([1.02,0,0.02,1],transform=ax2.transAxes)
fig.colorbar(Im2,
        extend='neither',
        cax=cax2,
    ).set_label(label='VDR', size=15, labelpad=6)
cax2.tick_params(labelsize=13)

### VDR distribution

In [ ]:
plt.hist(ds['vdr_cm'].values.flatten(), bins=50)

In [ ]:
plt.hist(ds['vdr_cm'].values.flatten(), bins=50,cumulative=True)

In [ ]:
np.nanmax(ds['vdr_cm'].values)

### Filtering by temperature regime

In [ ]:
clouds = ds.where(ds['cloud_mask'] == 1)

cold = clouds.where(clouds['temp'] < -40)
mixed = clouds.where((clouds['temp'] >= -40) & (clouds['temp'] <= 0))
warm = clouds.where(clouds['temp'] > 0)

In [ ]:
(_, temp_bins, _) = plt.hist(clouds['temp'].values.flatten(),bins=50)

In [ ]:
plt.hist(cold['temp'].values.flatten(), bins=temp_bins)
plt.hist(mixed['temp'].values.flatten(), bins=temp_bins)
plt.hist(warm['temp'].values.flatten(), bins=temp_bins)

In [ ]:
(_, vdr_bins, _) = plt.hist(ds['vdr_cm'].values.flatten(), bins=50)

In [ ]:
plt.hist(cold['vdr_cm'].values.flatten(), bins=vdr_bins)
plt.hist(mixed['vdr_cm'].values.flatten(), bins=vdr_bins)
plt.hist(warm['vdr_cm'].values.flatten(), bins=vdr_bins)

In [ ]:
plt.hist(cold['vdr_cm'].values.flatten(), bins=vdr_bins,cumulative=True, alpha=0.4)
plt.hist(mixed['vdr_cm'].values.flatten(), bins=vdr_bins,cumulative=True, alpha=0.4)
plt.hist(warm['vdr_cm'].values.flatten(), bins=vdr_bins,cumulative=True, alpha=0.4)

In [ ]:
fig = plt.figure()
ax = fig.subplots(1, 1)

cold_vdr = cold['vdr_cm'].values.flatten()[~np.isnan(cold['vdr_cm'].values.flatten())]
mixed_vdr = mixed['vdr_cm'].values.flatten()[~np.isnan(mixed['vdr_cm'].values.flatten())]
warm_vdr = warm['vdr_cm'].values.flatten()[~np.isnan(warm['vdr_cm'].values.flatten())]

ax.ecdf(cold_vdr, label='COLD < -40')
ax.ecdf(mixed_vdr, label='MIXED')
ax.ecdf(warm_vdr, label='WARM > 0')
ax.legend()

### CDF plots by temperature bin

In [ ]:
clouds = ds.where(ds['cloud_mask'] == 1)

temp_bins = [-50,-40,-30,-20,-10,0,10]

temps_gb = clouds.groupby_bins('temp', bins=temp_bins)

fig = plt.figure()
ax = fig.subplots(1, 1)

for (interval,group_ds) in temps_gb:
    vdr = group_ds['vdr_cm'].values.flatten()[~np.isnan(group_ds['vdr_cm'].values.flatten())]
    ax.ecdf(vdr, label=interval)

ax.legend()

ax.set_xlim(0,1)

plt.show()

## For larger set

In [ ]:
files = sorted(glob.glob(vdr_mask_data_dir.format('*')+ '/*.nc'))

ds_list = [xr.open_dataset(infile, chunks={"time": 10}) for infile in files]

In [ ]:
ds = xr.concat(ds_list, dim='time')
ds

In [ ]:
clouds = ds.where(ds['cloud_mask'] == 1)

### 1d hists of VDR and temps

In [ ]:
vdr_all = clouds['vdr_cm'].values.flatten()
temps_all = clouds['temp'].values.flatten()

In [ ]:
fig = plt.figure()
ax = fig.subplots(1, 1)

ax.hist(vdr_all, bins=50)

ax.set_xlim(0,1)
ax.set_xlabel('VDR')
ax.set_ylabel('Frequency')

# plt.savefig('/home/lukeew/Local_work/01_SLW_xgb/output/figs/vdr_hist.png', dpi=300)

In [ ]:
fig = plt.figure()
ax = fig.subplots(1, 1)

ax.hist(vdr_all, bins=50,cumulative=True, density=True)
vdr = vdr_all[~np.isnan(vdr_all)]
ax.ecdf(vdr)

ax.set_xlim(0,1)
ax.set_xlabel('VDR')
ax.set_ylabel('Cumulative Frequency')

# plt.savefig('/home/lukeew/Local_work/01_SLW_xgb/output/figs/vdr_cdf.png', dpi=300)

In [ ]:
fig = plt.figure()
ax = fig.subplots(1, 1)

ax.hist(temps_all,bins=50)
ax.vlines(0,0,30000,colors='red',linestyles='dashed')

ax.set_xlabel('Temperature ($\degree$C)')
ax.set_ylabel('Frequency')

# plt.savefig('/home/lukeew/Local_work/01_SLW_xgb/output/figs/cld_temps_hist.png', dpi=300)

### VDR CDFs with temperature grouping

In [ ]:
temp_bins = [-60,-50,-40,-30,-20,-10,0,10,20]

temps_gb = clouds.groupby_bins('temp', bins=temp_bins)

vdr_by_temp = {}

for (interval,group_ds) in temps_gb:
    vdr = group_ds['vdr_cm'].values.flatten()
    vdr = vdr[~np.isnan(vdr)]
    vdr_by_temp[interval] = vdr


In [ ]:
fig = plt.figure()
ax = fig.subplots(1, 1)

# colors = iter(plt.cm.jet_r(np.linspace(0,1,8)))
colors = iter(plt.cm.coolwarm([0,0.1,0.2,0.3, 0.4, 0.8, 0.9, 1]))


for (interval,vdr_arr) in vdr_by_temp.items():
    # label = str(interval) + '\nn={:,}'.format(len(vdr_arr))
    label = str(interval)

    ax.ecdf(vdr_arr, label=label, c=next(colors))    


ax.legend(title='Temperature\nRanges ($\degree$C)')

ax.set_xlim(0,1)

ax.set_xlabel('VDR')
ax.set_ylabel('Cumulative Frequency')

ax.set_xticks(np.linspace(0,1,11))
for label in ax.xaxis.get_ticklabels()[1::2]:
    label.set_visible(False)

ax_t = ax.secondary_xaxis('top')
ax_t.tick_params(axis='x', direction='inout',labeltop=False)
ax_t.set_xticks(np.linspace(0,1,11))
for label in ax_t.xaxis.get_ticklabels()[1::2]:
    label.set_visible(False)

plt.show()
# plt.savefig('/home/lukeew/Local_work/01_SLW_xgb/output/figs/vdr_cdf_temps.png', dpi=300)

### 2d hist of VDR and temps

In [ ]:
fig = plt.figure(figsize=(6,5))
ax = fig.subplots(1, 1)

norm = 'log'
ranges = [[0,1], [np.nanmin(temps_all),np.nanmax(temps_all)]]

counts, xedges, yedges, im = ax.hist2d(vdr_all, temps_all, bins=50, range=ranges,norm=norm)

fig.colorbar(im, ax=ax, label='Frequency')

ax.set_xlabel('VDR')
ax.set_ylabel('Temperature ($\degree$C)')

ax.vlines([0.1, 0.4], ranges[1][0], ranges[1][1], colors='red',linestyles='dashed')
ax.hlines(0, 0,1, colors='red',linestyles='dashed')

# plt.savefig('/home/lukeew/Local_work/01_SLW_xgb/output/figs/vdr_temp_2d_hist.png', dpi=300)

In [ ]:
## Labelled version ??

# fig = plt.figure(figsize=(8,5))
# ax = fig.subplots(1, 1)

# norm = 'log'
# ranges = [[0,0.8], [-40,20]]

# counts, xedges, yedges, im = ax.hist2d(vdr_all, temps_all, bins=50, range=ranges,norm=norm)

# fig.colorbar(im, ax=ax, label='Frequency')

# ax.set_xlabel('VDR')
# ax.set_ylabel('Temperature ($\degree$C)')

# ax.vlines([0.1, 0.4], ranges[1][0], ranges[1][1], colors='red',linestyles='dashed')
# ax.hlines(0, 0,1, colors='red',linestyles='dashed')

# ax.text(0.02, 10, 'WLWC', c='r', fontsize=10)
# ax.text(0.02, -20, 'SLWC', c='r', fontsize=10)
# ax.text(0.06, -30, 'SLCC', c='r', fontsize=16)
# ax.text(0.6, -20, 'IC', c='r', fontsize=16)
# ax.text(0.3, 10, 'Unknown', c='r', fontsize=14)